# 🛰️ Module 18: Model Evaluation, Robustness & Uncertainty Quantification

Welcome to **Module 18** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **Bayesian Deep Learning via Monte Carlo Dropout**: Perform $T$ stochastic forward passes at inference to quantify epistemic model ambiguity.
2. **Spatial Uncertainty Mapping**: Compute pixel-wise predictive standard deviations $\sigma(x, y)$ and distinguish high-confidence canopy loss from borderline ambiguous clearing.
3. **Sensor Noise Stress Testing**: Evaluate degradation under additive Gaussian sensor noise $\mathcal{N}(0, \sigma^2)$ representing sensor aging and thermal interference.
4. **Atmospheric Hazing & Cloud Occlusion**: Measure performance decay under synthetic cirrus cloud haze and optical attenuation.
5. **Calibration Curves & Expected Calibration Error (ECE)**: Construct reliability diagrams to assess whether predicted softmax probabilities accurately represent empirical correctness.

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_18_uncertainty_robustness.mc_dropout_evaluator import MCDropoutChangeDetector, estimate_epistemic_uncertainty
from modules.module_18_uncertainty_robustness.robustness_stress_tester import RobustnessStressTester, compute_calibration_curve

%matplotlib inline
print(f"✅ PyTorch version: {torch.__version__} | Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Load Pretrained Bayesian Siamese Architecture

We instantiate the `MCDropoutChangeDetector` with `dropout_prob=0.25` enabled in the bottleneck and decoder blocks.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MCDropoutChangeDetector(in_channels=5, base_features=16, dropout_prob=0.25).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Inspect Generated Robustness & Calibration Visualizations

Let's visualize the results produced by the master runner in `outputs/module_18/`.

In [ ]:
from PIL import Image

fig, axes = plt.subplots(3, 1, figsize=(14, 18))

img1 = Image.open('../outputs/module_18/01_mc_dropout_uncertainty_maps.png')
axes[0].imshow(img1)
axes[0].axis('off')
axes[0].set_title("1. Monte Carlo Dropout Epistemic Uncertainty & Ambiguity Mask", fontsize=13, fontweight='bold')

img2 = Image.open('../outputs/module_18/02_robustness_stress_test_curves.png')
axes[1].imshow(img2)
axes[1].axis('off')
axes[1].set_title("2. Robustness Stress Testing: Gaussian Noise & Cloud Haze Degradation", fontsize=13, fontweight='bold')

img3 = Image.open('../outputs/module_18/03_calibration_reliability_diagram.png')
axes[2].imshow(img3)
axes[2].axis('off')
axes[2].set_title("3. Probability Calibration Reliability Diagram & ECE", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Robustness & Uncertainty Quantification Report

Let's review the JSON audit report summarizing the quantitative findings.

In [ ]:
import json
report_path = '../outputs/module_18/robustness_and_uncertainty_report.json'
if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        report = json.load(f)
    print(json.dumps(report, indent=2))
else:
    print("Run run_module_18.py to generate report.")